Conab last_report download

In [12]:
import requests
from bs4 import BeautifulSoup
import re

def download_latest_coffee_report():
    base_url = "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/safra-de-cafe"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    try:
        print("Accessing the Conab webpage...")
        response = requests.get(base_url, headers=headers)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        links = soup.find_all('a', href=True)

        pdf_url = None
        report_name = ""

        target_pattern = re.compile(r"levantamento\s+de\s+caf[eé]", re.IGNORECASE)

        for link in links:
            link_text = link.get_text(separator=" ", strip=True)

            if target_pattern.search(link_text):
                pdf_url = link['href']
                report_name = re.sub(r'\s+', ' ', link_text)
                print(f"Latest report found: {report_name}")
                break

        if not pdf_url:
            print("Could not find the coffee harvest report link on the page.")
            return

        if not pdf_url.lower().endswith('.pdf'):
            print("Accessing the report page to extract the final PDF link...")
            internal_res = requests.get(pdf_url, headers=headers)
            internal_res.raise_for_status()
            internal_soup = BeautifulSoup(internal_res.text, 'html.parser')

            pdf_link = internal_soup.find('a', href=lambda href: href and href.lower().endswith('.pdf'))

            if pdf_link:
                pdf_url = pdf_link['href']
            else:
                print("PDF file not found on the report page.")
                return

        print(f"Starting download for: {pdf_url}")
        pdf_response = requests.get(pdf_url, headers=headers)
        pdf_response.raise_for_status()

        # FIX: Sanitize the report name for valid OS file naming and format as an f-string
        safe_report_name = re.sub(r'[\\/*?:"<>|]', "", report_name).strip()
        file_name = f"{safe_report_name}.pdf"

        with open(file_name, 'wb') as f:
            f.write(pdf_response.content)

        print(f"Download completed successfully! The report was saved as '{file_name}'.")

    except Exception as e:
        print(f"An error occurred during execution: {e}")

if __name__ == "__main__":
    download_latest_coffee_report()

Accessing the Conab webpage...
Latest report found: 1º Levantamento de Café - Safra 2026
Accessing the report page to extract the final PDF link...
Starting download for: https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/safra-de-cafe/1o-levantamento-de-cafe-safra-2026/boletim-de-safras-cafe-fevereiro-26.pdf
Download completed successfully! The report was saved as '1º Levantamento de Café - Safra 2026.pdf'.
